# FraudMind — Version 0 Demo

Runs the ATO Agent on the three canonical test cases defined in CLAUDE.md.

| Case | Description | Expected verdict |
|------|-------------|------------------|
| 1 | Clear ATO — new device, geo mismatch, email change, payout attempt | `block` |
| 2 | Clean legitimate login — known device, home geo, no sensitive actions | `allow` |
| 3 | Ambiguous — new device, known travel destination, no changes | `step_up` |

In [ ]:
import sys
sys.path.insert(0, '..')  # make src/ importable from notebooks/

from src.schemas.ato_schemas import MockATOSignals
from src.agents.ato_agent import run_ato_agent

In [ ]:
def print_verdict(case_label: str, signals: MockATOSignals) -> None:
    """Run the ATO agent and pretty-print the verdict."""
    print(f"{'='*60}")
    print(f"  {case_label}")
    print(f"{'='*60}")
    verdict = run_ato_agent(signals)
    print(f"  Verdict:            {verdict.verdict.upper()}")
    print(f"  Confidence:         {verdict.confidence:.0%}")
    print(f"  Primary signals:    {', '.join(verdict.primary_signals)}")
    print(f"  Reasoning:          {verdict.reasoning}")
    print(f"  Recommended action: {verdict.recommended_action}")
    print()

## Case 1 — Clear ATO
New device, login from Lagos (account home: San Jose), email change within 3 min, payout request.

In [ ]:
case1 = MockATOSignals(
    account_id="acc_001",
    account_age_days=210,
    is_new_device=True,
    device_id="dev_unknown_001",
    login_geo="Lagos, Nigeria",
    account_home_geo="San Jose, CA",
    geo_mismatch=True,
    time_since_last_login_days=2,
    immediate_high_value_action=True,
    email_change_attempted=True,
    password_change_attempted=False,
    support_ticket_language_match=False,
)

print_verdict("Case 1 — Clear ATO  (expected: BLOCK)", case1)

## Case 2 — Clean Legitimate Login
Known device (used 47 times), San Jose login matches home geo, no sensitive actions.

In [ ]:
case2 = MockATOSignals(
    account_id="acc_002",
    account_age_days=847,
    is_new_device=False,
    device_id="dev_trusted_047",
    login_geo="San Jose, CA",
    account_home_geo="San Jose, CA",
    geo_mismatch=False,
    time_since_last_login_days=1,
    immediate_high_value_action=False,
    email_change_attempted=False,
    password_change_attempted=False,
    support_ticket_language_match=True,
)

print_verdict("Case 2 — Clean Login  (expected: ALLOW)", case2)

## Case 3 — Ambiguous
New device, London login (account logged in from London twice before), no changes, no high-value action.

In [ ]:
case3 = MockATOSignals(
    account_id="acc_003",
    account_age_days=430,
    is_new_device=True,
    device_id="dev_unknown_002",
    login_geo="London, UK",
    account_home_geo="New York, NY",
    geo_mismatch=True,
    time_since_last_login_days=12,
    immediate_high_value_action=False,
    email_change_attempted=False,
    password_change_attempted=False,
    support_ticket_language_match=True,
)

print_verdict("Case 3 — Ambiguous  (expected: STEP_UP)", case3)

## Summary

All three cases above demonstrate the ATO Agent reasoning correctly across the risk spectrum:
- **Block** when multiple strong ATO signals converge
- **Allow** when all signals point to a trusted session
- **Step up** when the picture is ambiguous and more verification is warranted